In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "openai-community/roberta-base-openai-detector"

print(f"Loading {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

print(f"Model loaded on {device}")


# SCORING FUNCTION

def compute_openai_roberta_score(text, model, tokenizer, device, max_len=512):

    try:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            padding=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        ai_prob = probs[0][0].item()

        return ai_prob

    except Exception as e:
        return 0.5

# PREPARE FUNCTION

def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER

def evaluate_openai_roberta(texts, labels, model, tokenizer, device):

    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = compute_openai_roberta_score(t, model, tokenizer, device)
        scores.append(s)

    thr = np.median(scores)
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET
print("\nLoading BEEMO dataset...")
df = load_dataset("toloka/beemo")['train'].to_pandas()
print(f"Loaded {len(df)} samples")


# DEFINE TASKS
tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}


# RUN ALL TASKS
PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_openai_roberta(X, y, model, tokenizer, device)
    res[name] = mt

    # Save results
    model_name_clean = MODEL_NAME.split('/')[-1]
    pd.DataFrame({
        'text': X,
        'label': y,
        'score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_{model_name_clean}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")


# SUMMARY
summary = pd.DataFrame(res).T
model_name_clean = MODEL_NAME.split('/')[-1]
summary.to_csv(f"{PROJECT}/results/{model_name_clean}_summary.csv")

print("\n" + "="*60)
print(f"SUMMARY")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results saved to {PROJECT}/results/")


Loading openai-community/roberta-base-openai-detector


Some weights of the model checkpoint at openai-community/roberta-base-openai-detector were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Model loaded on cuda

Loading BEEMO dataset...
Loaded 2187 samples

 H_vs_M
Samples: 4374 (2187/2187)


Scoring: 100%|█████████████████████████████| 4374/4374 [00:35<00:00, 122.44it/s]


AUROC: 0.6701 | Acc: 0.6095 | F1: 0.6095

 E_vs_M
Samples: 4374 (2187/2187)


...

AUROC: 0.5276 | Acc: 0.5001 | F1: 0.6001

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|█████████████████████████████| 8748/8748 [01:10<00:00, 124.27it/s]


AUROC: 0.6108 | Acc: 0.5552 | F1: 0.4070

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|█████████████████████████████| 8748/8748 [01:10<00:00, 124.92it/s]


AUROC: 0.5738 | Acc: 0.5296 | F1: 0.6237

SUMMARY
         auroc  accuracy      f1
H_vs_M  0.6701    0.6095  0.6095
E_vs_M  0.6664    0.6141  0.6141
H_vs_E  0.4950    0.4856  0.4856
L_vs_M  0.6494    0.5765  0.4353
H_vs_L  0.5276    0.5001  0.6001
G_vs_M  0.6108    0.5552  0.4070
H_vs_G  0.5738    0.5296  0.6237

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/
